# Initialization

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
from pyspark.sql.functions import col, sum, when, upper, trim
from pyspark.sql import functions as F

In [0]:
def to_snake_case(column_name):
    column_name = unicodedata.normalize("NFKD", column_name)
    column_name = column_name.encode("ascii", "ignore").decode("utf-8")
    column_name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", column_name)
    column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")

    return column_name.lower()

In [0]:
def validate_codes(df, col_name, dim_table, dim_code="code", critical=False):

    dim = spark.table(dim_table)

    total_count = df.count()

    # Detect invalid values
    invalid = df.join(
        dim,
        df[col_name] == dim[dim_code],
        "left_anti"
    )

    invalid_count = invalid.count()

    # porcentaje
    invalid_pct = (invalid_count / total_count) * 100 if total_count > 0 else 0

    print(f"{col_name}: {invalid_count} invalid values ({invalid_pct:.2f}%)")

    # If critical → remove invalid rows
    if critical:
        df = df.join(
            dim,
            df[col_name] == dim[dim_code],
            "inner"
        )
        return df, invalid

    # If not critical → set invalids to NULL
    valid_values = [row[dim_code] for row in dim.collect()]

    df = df.withColumn(
        col_name,
        F.when(
            F.col(col_name).isin(valid_values),
            F.col(col_name)
        ).otherwise(F.lit(None))
    )

    return df, invalid

# Visualitation

In [0]:
df = spark.table("covid19.bronze.ine_deaths")
print(df) 

In [0]:
df.head()

# Cleaning

## Columns normalitation

In [0]:
df = df.toDF(*[to_snake_case(col) for col in df.columns])

## Delete exactly duplicate rows

In [0]:
# delete exact duplicate rows
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print(f"Total filas: {total_rows}")
print(f"Filas únicas: {unique_rows}")
print(f"Duplicados: {total_rows - unique_rows}")

## delete nulls 

In [0]:


null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
total_rows = df.count()

for column in df.columns:
    nulls = df.filter(col(column).isNull()).count()
    percentage = (nulls / total_rows) * 100

    print(
        f"{column}: {nulls} nulls ({percentage:.2f}%)"
    )

In [0]:
df = df.fillna({
    "Escodif": 99,
    "Ciuodif": 99
})

## correct data types

In [0]:
df.printSchema()

## standarize 

In [0]:
display(
    df.select("Caudef")
      .distinct()
      .orderBy("Caudef")
)

In [0]:

df = df.withColumn(
    "Caudef",
    upper(trim("Caudef"))
)

## semantic validation

In [0]:
display(df.select("Sexo").distinct())
display(df.select("Mesreg").distinct())
display(df.select("Edadif").describe())

In [0]:
df = df.withColumn(
    "Edadif",
    when(col("Edadif") == 999, None)
    .otherwise(col("Edadif"))
)

In [0]:
df.filter(df.Edadif < 0).count()

In [0]:
df = df.filter((df.Edadif >= 0) | df.Edadif.isNull())

## delete outlayers

In [0]:
q1, q3 = df.approxQuantile("Edadif", [0.25, 0.75], 0.01)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(lower, upper)

In [0]:
df = df.filter(
    (df.Edadif.isNull()) |
    ((df.Edadif >= lower) & (df.Edadif <= upper))
)

In [0]:
validation_config = {

    # Asistencia médica recibida
    "Asist": {
        "table": "covid19.metadata.dim_asistencia_recibida",
        "critical": False
    },

    # Certificador de defunción
    "Cerdef": {
        "table": "covid19.metadata.dim_certificador",
        "critical": True
    },

    # Lugar de ocurrencia
    "Ocur": {
        "table": "covid19.metadata.dim_citio_ocurrencia",
        "critical": True
    },

    # Geografía
    "Depreg": {
        "table": "covid19.metadata.dim_departamentos",
        "critical": True
    },
    "Depocu": {
        "table": "covid19.metadata.dim_departamentos",
        "critical": True
    },
    "Dredif": {
        "table": "covid19.metadata.dim_departamentos",
        "critical": False
    },

    "Mupreg": {
        "table": "covid19.metadata.dim_municipios",
        "critical": True
    },
    "Mupocu": {
        "table": "covid19.metadata.dim_municipios",
        "critical": True
    },
    "Mredif": {
        "table": "covid19.metadata.dim_municipios",
        "critical": False
    },

    # Escolaridad
    "Escodif": {
        "table": "covid19.metadata.dim_escolaridad",
        "critical": False
    },

    # Estado civil
    "Ecidif": {
        "table": "covid19.metadata.dim_estado_civil",
        "critical": False
    },

    # Ocupación
    "Ciuodif": {
        "table": "covid19.metadata.dim_ocupaciones",
        "critical": False
    },

    # Países / nacionalidad / residencia
    "Pnadif": {
        "table": "covid19.metadata.dim_paises",
        "critical": False
    },
    "Nacdif": {
        "table": "covid19.metadata.dim_paises",
        "critical": False
    },
    "Predif": {
        "table": "covid19.metadata.dim_paises",
        "critical": False
    },

    # Periodo de edad
    "Perdif": {
        "table": "covid19.metadata.dim_periodo_edad",
        "critical": False
    }
}

In [0]:
results = {}

df_clean = df  # copia de trabajo

for col_name, config in validation_config.items():

    df_clean, invalid_df = validate_codes(
        df_clean,
        col_name,
        config["table"],
        critical=config["critical"]
    )

    results[col_name] = invalid_df

    print(f"{col_name} -> invalid rows: {invalid_df.count()}")